In [ ]:
import os, os.path as op, glob
from copy import deepcopy
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from nilearn.plotting import plot_stat_map
from nilearn.datasets import load_mni152_template
from nilearn.glm.first_level import make_first_level_design_matrix

# Fill-in the path to the project folder
data_path = os.path.abspath("")
fig_dir = op.join(data_path, 'figures')
os.makedirs(fig_dir, exist_ok=True)

In [ ]:
img = nib.load(op.join(data_path, 'derivatives', 'tfMRI_LANGUAGE_LR_moco_smoothed-8mm.nii.gz'))
affine = img.affine
data = np.asanyarray(img.dataobj)
data.shape

In [ ]:
# Make variables:
# 'vol_shape' for shape of volumes
# 'n_vols' for number of volumes
vol_shape = np.array(data.shape[:3])
n_vols = data.shape[3]

In [ ]:
# Make variables:
# 'vol_shape' for shape of volumes
# 'n_vols' for number of volumes
vol_shape = np.array(data.shape[:3])
n_vols = data.shape[3]

In [ ]:
# Note: In our case the background is encoded as <= 500
# you can consider that the first volume's background
# voxels are the same as all following volumes
slice_non_background = data[..., 0] > 500
# Vectorize : Taking only non-zero voxels into a vector
# (NOTE: that the order is important)
samples = data[slice_non_background]
samples.shape

In [ ]:
# Calculate the mean across columns
spatial_means = samples.mean(axis=1)
# Row means copied n_vols times so that we substract for each timepoint the spatial mean
row_means = np.tile(spatial_means[:, None], (1, n_vols))
# Subtract the means for each row, put the result into X
X = samples - row_means



In [ ]:
X.shape

In [ ]:

X_kmeans = deepcopy(X)

In [ ]:
max_nb_clusters = 16
avg_dist_samples = []
for nb_cluster in range(1, max_nb_clusters):
    kmeans = KMeans(n_clusters=nb_cluster, random_state=0, n_init="auto").fit(X_kmeans.T)
    avg_dist_samples.append(kmeans.inertia_)

In [ ]:
fig, ax = plt.subplots(1, figsize=(7,5))

ax.plot(np.arange(1, len(avg_dist_samples)+1), avg_dist_samples, label='WCSS', c='k')

ax.set_xlabel('Clusters #', size=15)
ax.legend(prop={'size':15})
ax.tick_params(axis='both', which='major', labelsize=15)
ax.grid(True, alpha=0.3)
fig.savefig(op.join(fig_dir, 'kmeans_elbow.png'), dpi=300, bbox_inches='tight')

In [ ]:
nb_clusters = 10 # The number of clusters you want (choose it from the elbow plot)
kmeans = KMeans(n_clusters=nb_clusters, random_state=0, n_init="auto").fit(X_kmeans.T)

kmeans_clusters = [] # List of spatial components (you should have in the list volumes)
for center in kmeans.cluster_centers_:
    vol = np.zeros(tuple(vol_shape))
    vol[slice_non_background] = center
    kmeans_clusters.append(vol)

In [ ]:
for visual_idx in range(nb_clusters):
    plot_stat_map(nib.Nifti1Image(kmeans_clusters[visual_idx], affine), bg_img=load_mni152_template(),
                  threshold=1e-3, display_mode='z', cut_coords=[49,37,00], black_bg=True,
                  title=f'K-means Cluster {visual_idx}')
    plt.savefig(op.join(fig_dir, f'kmeans_cluster_{visual_idx}.png'), dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Correlation with GLM regressors
tr = float(img.header.get_zooms()[3])
start_vol = 1 # Number of volumes removed in the functional preprocessing

events = pd.read_csv(glob.glob(op.join(data_path, '**', 'tfMRI_LANGUAGE_LR_task_onsets.csv'), recursive=True)[0])
# Shift the onsets because the first volumes were removed
events['onset'] = events['onset'] - start_vol * tr

frame_times = np.arange(n_vols) * tr
design_mat = make_first_level_design_matrix(frame_times, events, hrf_model='spm', drift_model=None)
regressors = [c for c in design_mat.columns if c != 'constant']

In [ ]:


# Binary time course of cluster membership (1 if timepoint belongs to cluster c, 0 otherwise)
binary_tc = pd.DataFrame({f'Cluster {c}': (kmeans.labels_ == c).astype(float)
                          for c in range(min(10, nb_clusters))})

# Correlation between cluster time courses and GLM regressors
corr = pd.concat([binary_tc, design_mat[regressors].reset_index(drop=True)], axis=1).corr()
corr = corr.loc[binary_tc.columns, regressors]

plt.figure(figsize=(12, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1, center=0)
plt.title('Cluster time courses vs Task Events')
plt.tight_layout()
plt.savefig(op.join(fig_dir, 'kmeans_glm_correlation.png'), dpi=300, bbox_inches='tight')
plt.show()